In [ ]:
# NLGCL Kaggle Notebook
# This notebook runs the NLGCL model on the Kaggle platform.
# It automatically sets up the environment, downloads the dataset, and runs training.

import os
import sys
import shutil
import subprocess

# 1. Setup Environment & Clone
# Check if NLGCL exists, if not clone it.
if not os.path.exists('NLGCL') and not os.path.exists('recbole_gnn'):
    print("Cloning repository...")
    !git clone https://github.com/Jinfeng-Xu/NLGCL.git
    os.chdir('NLGCL')
else:
    if os.path.basename(os.getcwd()) != 'NLGCL' and os.path.exists('NLGCL'):
        os.chdir('NLGCL')
    print("Repository already exists. Updating...")
    !git pull

sys.path.append(os.getcwd())
print(f"Working Directory: {os.getcwd()}")

# 1.5 Patch Code and Clean Environment
# Force-remove cached bytecode to prevent loading old, broken versions
print("Cleaning pycache...")
subprocess.run(["find", ".", "-name", "__pycache__", "-type", "d", "-exec", "rm", "-rf", "{}", "+"])
subprocess.run(["find", ".", "-name", "*.pyc", "-delete"])

# Patch 1: recbole_gnn/config.py for NumPy 2.0 Compatibility
patch_content_config = r'''import os
import recbole
from recbole.config.configurator import Config as RecBole_Config
from recbole.utils import ModelType as RecBoleModelType
import numpy as np

# Global patch for NumPy 2.0 immediately
if not hasattr(np, 'bool'): np.bool = bool
if not hasattr(np, 'int'): np.int = int
if not hasattr(np, 'float'): np.float = float
if not hasattr(np, 'complex'): np.complex = complex
if not hasattr(np, 'object'): np.object = object
if not hasattr(np, 'str'): np.str = str
if not hasattr(np, 'long'): np.long = int
if not hasattr(np, 'unicode'): np.unicode = str

from recbole_gnn.utils import get_model, ModelType

class Config(RecBole_Config):
    def __init__(self, model=None, dataset=None, config_file_list=None, config_dict=None):
        if recbole.__version__ >= "1.1.1":
            self.compatibility_settings()
        super(Config, self).__init__(model, dataset, config_file_list, config_dict)

    def compatibility_settings(self):
        import numpy as np
        # Redundant safety checks
        if not hasattr(np, 'bool'): np.bool = bool
        if not hasattr(np, 'int'): np.int = int
        if not hasattr(np, 'float'): np.float = float
        if not hasattr(np, 'complex'): np.complex = complex
        if not hasattr(np, 'object'): np.object = object

    def _get_model_and_dataset(self, model, dataset):
        if model is None:
            try:
                model = self.external_config_dict['model']
            except KeyError:
                raise KeyError(
                    'model need to be specified in at least one of the these ways: '
                    '[model variable, config file, config dict, command line] '
                )
        if not isinstance(model, str):
            final_model_class = model
            final_model = model.__name__
        else:
            final_model = model
            final_model_class = get_model(final_model)

        if dataset is None:
            try:
                final_dataset = self.external_config_dict['dataset']
            except KeyError:
                raise KeyError(
                    'dataset need to be specified in at least one of the these ways: '
                    '[dataset variable, config file, config dict, command line] '
                )
        else:
            final_dataset = dataset

        return final_model, final_model_class, final_dataset

    def _load_internal_config_dict(self, model, model_class, dataset):
        super()._load_internal_config_dict(model, model_class, dataset)
        current_path = os.path.dirname(os.path.realpath(__file__))
        model_init_file = os.path.join(current_path, './properties/model/' + model + '.yaml')
        quick_start_config_path = os.path.join(current_path, './properties/quick_start_config/')
        sequential_base_init = os.path.join(quick_start_config_path, 'sequential_base.yaml')
        social_base_init = os.path.join(quick_start_config_path, 'social_base.yaml')

        if os.path.isfile(model_init_file):
            config_dict = self._update_internal_config_dict(model_init_file)

        self.internal_config_dict['MODEL_TYPE'] = model_class.type
        if self.internal_config_dict['MODEL_TYPE'] == RecBoleModelType.SEQUENTIAL:
            self._update_internal_config_dict(sequential_base_init)
        if self.internal_config_dict['MODEL_TYPE'] == ModelType.SOCIAL:
             self._update_internal_config_dict(social_base_init)
'''

# Patch 2: recbole_gnn/model/general_recommender/__init__.py to remove missing imports
patch_content_init = r'''# Only import the models that actually exist in the repo
try:
    from recbole_gnn.model.general_recommender.nlgcl import NLGCL
except ImportError:
    pass
'''

config_path = 'recbole_gnn/config.py'
init_path = 'recbole_gnn/model/general_recommender/__init__.py'

if os.path.exists('recbole_gnn'):
    print(f"Patching {config_path} for NumPy 2.0...")
    with open(config_path, 'w') as f:
        f.write(patch_content_config)
    
    if os.path.exists(os.path.dirname(init_path)):
        print(f"Patching {init_path} to remove missing imports...")
        with open(init_path, 'w') as f:
            f.write(patch_content_init)
            
    print("Patches applied.")
else:
    print("WARNING: recbole_gnn directory not found. Patch skipped.")


# 2. Check and Install Dependencies
try:
    import recbole
    print(f"RecBole version {recbole.__version__} is detected.")
except ImportError:
    print("RecBole not found. Installing...")
    !pip install recbole
    # Also install other requirements
    if os.path.exists('requirements.txt'):
         !pip install -r requirements.txt
    
    # GNN libraries
    !pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

# 3. Dataset Preparation
# Use lowercase 'yelp' to match RecBole standard convention
dataset_name = 'yelp' 
target_dir = f'dataset/{dataset_name}'

# Check /kaggle/input for pre-loaded dataset
if not os.path.exists(target_dir) and os.path.exists('/kaggle/input'):
    print("Searching /kaggle/input for dataset...")
    for root, dirs, files in os.walk('/kaggle/input'):
        # Case insensitive search
        if f'{dataset_name}.inter' in [f.lower() for f in files]:
            print(f"Found in {root}. Copying...")
            shutil.copytree(root, target_dir, dirs_exist_ok=True)
            break
        # Check if directory matches
        dirname = os.path.basename(root).lower()
        if dirname == dataset_name:
             print(f"Found database folder {root}. Copying...")
             shutil.copytree(root, target_dir, dirs_exist_ok=True)
             break

# Download from S3 if still missing
if not os.path.exists(target_dir) or not any(f.endswith('.inter') for f in os.listdir(target_dir)):
    print(f"Downloading {dataset_name}...")
    import requests, zipfile
    # Original URL might use Capitalized Yelp in path, but zip content is usually lowercase
    url = "https://recbole.s3-accelerate.amazonaws.com/ProcessedDatasets/Yelp/yelp.zip"
    
    if not os.path.exists(target_dir): os.makedirs(target_dir)
    zip_path = os.path.join(target_dir, f"{dataset_name}.zip")
    
    try:
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(zip_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192): f.write(chunk)
        with zipfile.ZipFile(zip_path, 'r') as z: z.extractall(target_dir)
        os.remove(zip_path)
        print("Download complete.")
        
        # Verify extraction structure
        files = os.listdir(target_dir)
        print(f"Files in {target_dir}: {files}")
        
    except Exception as e:
        print(f"Download failed: {e}")

# 4. Run Training
import sys
# Force reload of modules to ensure patch is picked up
if 'recbole_gnn.config' in sys.modules:
    import importlib
    import recbole_gnn.config
    print("Reloading recbole_gnn.config...")
    importlib.reload(recbole_gnn.config)

import torch
from logging import getLogger
from recbole.utils import init_logger, init_seed, set_color
from recbole_gnn.config import Config
from recbole_gnn.utils import create_dataset, data_preparation, get_model, get_trainer

def run_single_model(args):
    # args.dataset must match the folder name and the .inter filename (e.g. 'yelp' -> dataset/yelp/yelp.inter)
    print(f"Running model with dataset: {args.dataset}")
    config = Config(model=args.model, dataset=args.dataset, config_file_list=args.config_file_list)
    
    # Ensure data path is correct
    if 'data_path' not in config: 
        config['data_path'] = 'dataset/' # This means it looks in dataset/{dataset_name}
    
    print(f"Config data_path: {config['data_path']}")

    init_seed(config['seed'], config['reproducibility'])
    init_logger(config)
    logger = getLogger()
    logger.info(config)

    dataset = create_dataset(config)
    logger.info(dataset)

    train_data, valid_data, test_data = data_preparation(config, dataset)

    model = get_model(config['model'])(config, train_data.dataset).to(config['device'])
    logger.info(model)

    trainer = get_trainer(config['MODEL_TYPE'], config['model'])(config, model)

    best_valid_score, best_valid_result = trainer.fit(train_data, valid_data, saved=True, show_progress=config['show_progress'])
    test_result = trainer.evaluate(test_data, load_best_model=True, show_progress=config['show_progress'])

    logger.info(set_color('best valid ', 'yellow') + f': {best_valid_result}')
    logger.info(set_color('test result', 'yellow') + f': {test_result}')

class Args:
    def __init__(self):
        self.model = 'NLGCL'
        self.dataset = 'yelp' # Changed to lowercase to match filename 'yelp.inter'
        self.config_files = ''
        self.config_file_list = ['properties/overall.yaml']

args = Args()
dataset_path = f'dataset/{args.dataset}'

# Handle capitalization mismatch if user has a folder 'Yelp' instead of 'yelp'
if not os.path.exists(dataset_path) and os.path.exists(f'dataset/{args.dataset.capitalize()}'):
    print(f"Renaming dataset/{args.dataset.capitalize()} to {dataset_path}")
    shutil.move(f'dataset/{args.dataset.capitalize()}', dataset_path)

if os.path.exists(dataset_path):
    run_single_model(args)
else:
    print(f"Dataset missing at {dataset_path}.")